In [2]:
# ------------------------------
# Importing necessary libraries
# ------------------------------

import pandas as pd
import tqdm

print("Successfully loaded all libraries!")

Successfully loaded all libraries!


In [3]:
# ------------------------------
# Loading cleaned football data
# ------------------------------

df = pd.read_csv('../data/processed/football_cleaned.csv', dtype={'season': str}, parse_dates=['Date'])

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns")
print(f"Date range: {df['Date'].min().date()} - {df['Date'].max().date()}")
print(f"\nLeagues: {df['league'].unique()}")
print(f"Seasons: {sorted(df['season'].unique())}")

Loaded 19833 rows and 30 columns
Date range: 2014-08-08 - 2025-05-25

Leagues: <StringArray>
['ligue_1', 'premier_league', 'bundesliga', 'la_liga', 'serie_a']
Length: 5, dtype: str
Seasons: ['1415', '1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425']


In [4]:
# -----------------
# Helper functions
# -----------------

def get_recent_team_stats(df, team, match_date, venue):
    if venue == 'home':
        team_matches = df[(df['HomeTeam'] == team) & (df['Date'] < match_date)]
    elif venue == 'away':
        team_matches = df[(df['AwayTeam'] == team) & (df['Date'] < match_date)]
    else:
        team_matches = df[((df['HomeTeam'] == team) | (df['AwayTeam'] == team)) & 
                          (df['Date'] < match_date)]
    
    return team_matches.sort_values('Date', ascending=False)

def calculate_points(result, is_home_team):
    if result == 'D':
        return 1
    elif (result == 'H' and is_home_team) or (result == 'A' and not is_home_team):
        return 3
    else:
        return 0
    
def get_team_form(df, team, match_date, n_matches, venue):
    team_matches = get_recent_team_stats(df, team, match_date, venue)
    
    if len(team_matches) == 0:
        return 0, 0, 0, 0, 0, 0, 0, 0
    
    recent_matches = team_matches.head(n_matches)
    points = wins = draws = losses = goals_for = goals_against = st_for = st_against = 0
    
    for _, match in recent_matches.iterrows():
        is_home = match['HomeTeam'] == team
        match_points = calculate_points(match['FTR'], is_home)
        points += match_points
        
        if match_points == 3:
            wins += 1
        elif match_points == 1:
            draws += 1
        else:
            losses += 1
        
        if is_home:
            goals_for += match['FTHG']
            goals_against += match['FTAG']
            st_for += match['HST']
            st_against += match['AST']
        else:
            goals_for += match['FTAG']
            goals_against += match['FTHG']
            st_for += match['AST']
            st_against += match['HST']
    
    return points, wins, draws, losses, goals_for, goals_against, st_for, st_against

def get_elo_ratings(elo_home, elo_away, result, k=20):
    exp_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))
    exp_away = 1 - exp_home

    if result == 'H':
        score_home, score_away = 1, 0
    elif result == 'D':
        score_home, score_away = 0.5, 0.5
    else:
        score_home, score_away = 0, 1

    elo_home = elo_home + k * (score_home - exp_home)
    elo_away = elo_away + k * (score_away - exp_away)

    return elo_home, elo_away

def update_attack_and_defense_elo(attack_elo_home, defense_elo_home,
                              attack_elo_away, defense_elo_away,
                              home_goals, away_goals, k=10):

    exp_home_goals = (attack_elo_home - defense_elo_away) / 400 + 1.5
    exp_away_goals = (attack_elo_away - defense_elo_home) / 400 + 1.2

    new_attack_elo_home = attack_elo_home + k * (home_goals - exp_home_goals)
    new_attack_elo_away = attack_elo_away + k * (away_goals - exp_away_goals)
    new_defense_elo_home = defense_elo_home + k * (exp_away_goals - away_goals)
    new_defense_elo_away = defense_elo_away + k * (exp_home_goals - home_goals)

    return new_attack_elo_home, new_defense_elo_home, new_attack_elo_away, new_defense_elo_away


In [8]:
# ---------------------
# Feature Engineering
# ---------------------

def create_match_features(df, match_idx):
    match = df.iloc[match_idx]
    home_team = match['HomeTeam']
    away_team = match['AwayTeam']
    match_date = match['Date']

    current_elo_home = elo[home_team]
    current_elo_away = elo[away_team]
    elo_diff = current_elo_home - current_elo_away
    current_attack_elo_home = attack_elo[home_team]
    current_defense_elo_home = defense_elo[home_team]
    current_attack_elo_away = attack_elo[away_team]
    current_defense_elo_away = defense_elo[away_team]

    features = {}
    
    # Home and away team features in the last 5 matches
    _, home_wins, home_draws, home_losses, home_goals_scored, home_goals_conceded, home_shots_for, home_shots_against = get_team_form(df, home_team, match_date, 5, 'any')
    _, away_wins, away_draws, away_losses, away_goals_scored, away_goals_conceded, away_shots_for, away_shots_against = get_team_form(df, away_team, match_date, 5, 'any')
    
    home_win_rate = home_wins / 5
    home_draw_rate = home_draws / 5
    home_loss_rate = home_losses / 5
    home_goal_diff = home_goals_scored - home_goals_conceded
    home_momentum = home_goal_diff + home_wins * 0.5
    home_avg_goals_scored = home_goals_scored / 5
    home_avg_goals_conceded = home_goals_conceded / 5
    home_avg_shots_for = home_shots_for / 5
    home_avg_shots_against = home_shots_against / 5

    away_win_rate = away_wins / 5
    away_draw_rate = away_draws / 5
    away_loss_rate = away_losses / 5
    away_goal_diff = away_goals_scored - away_goals_conceded
    away_momentum = away_goal_diff + away_wins * 0.5
    away_avg_goals_scored = away_goals_scored / 5
    away_avg_goals_conceded = away_goals_conceded / 5
    away_avg_shots_for = away_shots_for / 5
    away_avg_shots_against = away_shots_against / 5

    # Home team features in the last 5 home matches
    _, home_wins_home, home_draws_home, home_losses_home, home_goals_scored_home, home_goals_conceded_home, home_shots_for_home, home_shots_against_home = get_team_form(df, home_team, match_date, 5, 'home')
    home_win_rate_home = home_wins_home / 5
    home_draw_rate_home = home_draws_home / 5
    home_loss_rate_home = home_losses_home / 5
    home_goal_diff_home = home_goals_scored_home - home_goals_conceded_home
    home_momentum_home = home_goal_diff_home + home_wins_home * 0.5
    home_avg_goals_scored_home = home_goals_scored_home / 5
    home_avg_goals_conceded_home = home_goals_conceded_home / 5
    home_avg_shots_for_home = home_shots_for_home / 5
    home_avg_shots_against_home = home_shots_against_home / 5

    # Away team features in the last 5 away matches
    _, away_wins_away, away_draws_away, away_losses_away, away_goals_scored_away, away_goals_conceded_away, away_shots_for_away, away_shots_against_away = get_team_form(df, away_team, match_date, 5, 'away')
    away_win_rate_away = away_wins_away / 5
    away_draw_rate_away = away_draws_away / 5
    away_loss_rate_away = away_losses_away / 5
    away_goal_diff_away = away_goals_scored_away - away_goals_conceded_away
    away_momentum_away = away_goal_diff_away + away_wins_away * 0.5
    away_avg_goals_scored_away = away_goals_scored_away / 5
    away_avg_goals_conceded_away = away_goals_conceded_away / 5
    away_avg_shots_for_away = away_shots_for_away / 5
    away_avg_shots_against_away = away_shots_against_away / 5

    # ELO ratings
    new_elo_home, new_elo_away = get_elo_ratings(current_elo_home, current_elo_away, match['FTR'])
    elo[home_team] = new_elo_home
    elo[away_team] = new_elo_away
    
    new_attack_elo_home, new_defense_elo_home, new_attack_elo_away, new_defense_elo_away = update_attack_and_defense_elo(current_attack_elo_home, current_defense_elo_home, 
                                                                                                                     current_attack_elo_away, current_defense_elo_away,
                                                                                                                     match['FTHG'], match['FTAG'])
    attack_elo[home_team] = new_attack_elo_home
    defense_elo[home_team] = new_defense_elo_home
    attack_elo[away_team] = new_attack_elo_away
    defense_elo[away_team] = new_defense_elo_away


    features = {
        # Home team stats in last 5 matches
        'home_win_rate': home_win_rate,
        'home_draw_rate': home_draw_rate,
        'home_loss_rate': home_loss_rate,
        'home_goal_diff': home_goal_diff,
        'home_momentum': home_momentum,
        'home_avg_goals_scored': home_avg_goals_scored,
        'home_avg_goals_conceded': home_avg_goals_conceded,
        'home_avg_shots_for': home_avg_shots_for,
        'home_avg_shots_against': home_avg_shots_against,
        
        # Away team stats in last 5 matches
        'away_win_rate': away_win_rate,
        'away_draw_rate': away_draw_rate,
        'away_loss_rate': away_loss_rate,
        'away_goal_diff': away_goal_diff,
        'away_momentum': away_momentum,
        'away_avg_goals_scored': away_avg_goals_scored,
        'away_avg_goals_conceded': away_avg_goals_conceded,
        'away_avg_shots_for': away_avg_shots_for,
        'away_avg_shots_against': away_avg_shots_against,

        # Home team stats in last 5 home matches
        'home_win_rate_home': home_win_rate_home,
        'home_draw_rate_home': home_draw_rate_home,
        'home_loss_rate_home': home_loss_rate_home,
        'home_goal_diff_home': home_goal_diff_home,
        'home_momentum_home': home_momentum_home,
        'home_avg_goals_scored_home': home_avg_goals_scored_home,
        'home_avg_goals_conceded_home': home_avg_goals_conceded_home,
        'home_avg_shots_for_home': home_avg_shots_for_home,
        'home_avg_shots_against_home': home_avg_shots_against_home,

        # Away team stats in last 5 away matches
        'away_win_rate_away': away_win_rate_away,
        'away_draw_rate_away': away_draw_rate_away,
        'away_loss_rate_away': away_loss_rate_away,
        'away_goal_diff_away': away_goal_diff_away,
        'away_momentum_away': away_momentum_away,
        'away_avg_goals_scored_away': away_avg_goals_scored_away,
        'away_avg_goals_conceded_away': away_avg_goals_conceded_away,
        'away_avg_shots_for_away': away_avg_shots_for_away,
        'away_avg_shots_against_away': away_avg_shots_against_away,
                
        # Elo ratings
        'elo_home': current_elo_home,
        'elo_away': current_elo_away,
        'elo_diff': elo_diff,
        'attack_elo_home': current_attack_elo_home,
        'defense_elo_home': current_defense_elo_home,
        'attack_elo_away': current_attack_elo_away,
        'defense_elo_away': current_defense_elo_away
    }
    
    return features

In [9]:
# --------------------------------------------
# Applying feature engineering to all matches
# --------------------------------------------

print("Creating features for all matches...")
print("This may take 3-5 minutes...")
teams = set(df['HomeTeam']).union(set(df['AwayTeam']))
elo = {team: 1500 for team in teams}
attack_elo = {team: 1500 for team in teams}
defense_elo = {team: 1500 for team in teams}

features_list = []

for idx in tqdm.tqdm(range(len(df))):
    try:
        features = create_match_features(df, idx)
        features_list.append(features)
    except Exception as e:
        if features_list:
            empty_features = {key: 0 for key in features_list[0].keys()}
            features_list.append(empty_features)

df_features = pd.DataFrame(features_list)

print(f"\nFeatures created: {df_features.shape[0]} rows, {df_features.shape[1]} columns")
print(f"\nFeature columns:")
print(df_features.columns.tolist())
print(f"\nFirst row:")
print(df_features.iloc[0])
print(f"\nCheck for NaN values:")
print(df_features.isnull().sum().sum())

Creating features for all matches...
This may take 3-5 minutes...


  0%|          | 0/19833 [00:00<?, ?it/s]

100%|██████████| 19833/19833 [03:52<00:00, 85.42it/s]


Features created: 19833 rows, 43 columns

Feature columns:
['home_win_rate', 'home_draw_rate', 'home_loss_rate', 'home_goal_diff', 'home_momentum', 'home_avg_goals_scored', 'home_avg_goals_conceded', 'home_avg_shots_for', 'home_avg_shots_against', 'away_win_rate', 'away_draw_rate', 'away_loss_rate', 'away_goal_diff', 'away_momentum', 'away_avg_goals_scored', 'away_avg_goals_conceded', 'away_avg_shots_for', 'away_avg_shots_against', 'home_win_rate_home', 'home_draw_rate_home', 'home_loss_rate_home', 'home_goal_diff_home', 'home_momentum_home', 'home_avg_goals_scored_home', 'home_avg_goals_conceded_home', 'home_avg_shots_for_home', 'home_avg_shots_against_home', 'away_win_rate_away', 'away_draw_rate_away', 'away_loss_rate_away', 'away_goal_diff_away', 'away_momentum_away', 'away_avg_goals_scored_away', 'away_avg_goals_conceded_away', 'away_avg_shots_for_away', 'away_avg_shots_against_away', 'elo_home', 'elo_away', 'elo_diff', 'attack_elo_home', 'defense_elo_home', 'attack_elo_away', 'de

In [10]:
# --------------------------------------------------------------------
# Combining features with original data and creating datasets A and B
# --------------------------------------------------------------------

df_features.reset_index(drop=True, inplace=True)
df.reset_index(drop=True, inplace=True)
df_final = pd.concat([df[['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'league', 'season', 
                           'AvgH', 'AvgD', 'AvgA', 'MaxH', 'MaxD', 'MaxA']], 
                       df_features], axis=1)

# Dataset A - All seasons, without odds
odds_cols = ['AvgH', 'AvgD', 'AvgA', 'MaxH', 'MaxD', 'MaxA']
df_dataset_a = df_final.drop(columns=odds_cols)

print(f"\nDataset A (all seasons, no odds):")
print(f"  Shape: {df_dataset_a.shape}")
print(f"  Seasons: {sorted(df_dataset_a['season'].unique())}")
print(f"  Missing values: {df_dataset_a.isnull().sum().sum()}")

# Dataset B - Only seasons with odds (2019/20 onwards)
odds_seasons = ['1920', '2021', '2122', '2223', '2324', '2425']
df_dataset_b = df_final[df_final['season'].isin(odds_seasons)].copy()
df_dataset_b = df_dataset_b.dropna(subset=['AvgH', 'AvgD', 'AvgA', 'MaxH', 'MaxD', 'MaxA'])

print(f"\nDataset B (seasons with odds only):")
print(f"  Shape: {df_dataset_b.shape}")
print(f"  Seasons: {sorted(df_dataset_b['season'].unique())}")
print(f"  Missing values: {df_dataset_b.isnull().sum().sum()}")

df_dataset_a.to_csv('../data/processed/football_dataset_a_all_seasons.csv', index=False)
df_dataset_b.to_csv('../data/processed/football_dataset_b_with_odds.csv', index=False)


Dataset A (all seasons, no odds):
  Shape: (19833, 49)
  Seasons: ['1415', '1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425']
  Missing values: 0

Dataset B (seasons with odds only):
  Shape: (10705, 55)
  Seasons: ['1920', '2021', '2122', '2223', '2324', '2425']
  Missing values: 0
